In [16]:
# function that constructs a trip based on a group of timestamps for a vehicle_id
def construct_trip(group):
    group_sorted = group.sort_values("timestep_time")
    first_row = group_sorted.iloc[0]
    last_row = group_sorted.iloc[-1]

    return pd.Series(
        {
            "start_x": first_row["vehicle_x"],
            "start_y": first_row["vehicle_y"],
            "destination_x": last_row["vehicle_x"],
            "destination_y": last_row["vehicle_y"],
            "manhattan_distance": abs(last_row["vehicle_x"] - first_row["vehicle_x"])
            + abs(last_row["vehicle_y"] - first_row["vehicle_y"]),
            "euclidean_distance": np.linalg.norm(
                first_row[["vehicle_x", "vehicle_y"]] - last_row[["vehicle_x", "vehicle_y"]]
            ),
            "hour_of_day": group["hour_of_day"].mode()[0],
            "vehicle_type_bus": first_row["vehicle_type_bus"],
            "vehicle_type_car": first_row["vehicle_type_car"],
            "vehicle_type_ldv": first_row["vehicle_type_ldv"],
            "vehicle_type_truck": first_row["vehicle_type_truck"],
            "trip_duration": last_row["timestep_time"] - first_row["timestep_time"],
        }
    )

In [17]:
# group by vehicle_id and construct the trips dataframe
trips_train = fcd_train.groupby("vehicle_id").apply(construct_trip, include_groups=False).reset_index()
trips_test = fcd_test.groupby("vehicle_id").apply(construct_trip, include_groups=False).reset_index()

## Positional Encoding


In [18]:
# function that applies fourier positional encoding to a coordinate
def fourier_positional_encoding(x, num_freqs=4):
    freqs = np.array([2**i for i in range(num_freqs)])
    # for each frequency, compute sine and cosine
    encoded = [np.sin(freqs * x), np.cos(freqs * x)]
    return np.concatenate(encoded, axis=-1)

In [19]:
# apply to both start and destination coordinates, then add to dataframe
num_freqs = 4
start_x_enc = trips_train["start_x"].apply(lambda x: fourier_positional_encoding(x, num_freqs))
start_y_enc = trips_train["start_y"].apply(lambda x: fourier_positional_encoding(x, num_freqs))
dest_x_enc = trips_train["destination_x"].apply(lambda x: fourier_positional_encoding(x, num_freqs))
dest_y_enc = trips_train["destination_y"].apply(lambda x: fourier_positional_encoding(x, num_freqs))

# convert the series of np.arrays into a dataframe of features
start_x_enc_df = pd.DataFrame(np.stack(start_x_enc.values), columns=[f"start_x_enc_{i}" for i in range(2 * num_freqs)])
start_y_enc_df = pd.DataFrame(np.stack(start_y_enc.values), columns=[f"start_y_enc_{i}" for i in range(2 * num_freqs)])
dest_x_enc_df = pd.DataFrame(np.stack(dest_x_enc.values), columns=[f"dest_x_enc_{i}" for i in range(2 * num_freqs)])
dest_y_enc_df = pd.DataFrame(np.stack(dest_y_enc.values), columns=[f"dest_y_enc_{i}" for i in range(2 * num_freqs)])

# concatenate these encoding dataframes with the original dataframe
trips_train = pd.concat(
    [trips_train.reset_index(drop=True), start_x_enc_df, start_y_enc_df, dest_x_enc_df, dest_y_enc_df], axis=1
)

# apply to both start and destination coordinates, then add to dataframe
start_x_enc = trips_test["start_x"].apply(lambda x: fourier_positional_encoding(x, num_freqs))
start_y_enc = trips_test["start_y"].apply(lambda x: fourier_positional_encoding(x, num_freqs))
dest_x_enc = trips_test["destination_x"].apply(lambda x: fourier_positional_encoding(x, num_freqs))
dest_y_enc = trips_test["destination_y"].apply(lambda x: fourier_positional_encoding(x, num_freqs))

# convert the series of np.arrays into a dataframe of features
start_x_enc_df = pd.DataFrame(np.stack(start_x_enc.values), columns=[f"start_x_enc_{i}" for i in range(2 * num_freqs)])
start_y_enc_df = pd.DataFrame(np.stack(start_y_enc.values), columns=[f"start_y_enc_{i}" for i in range(2 * num_freqs)])
dest_x_enc_df = pd.DataFrame(np.stack(dest_x_enc.values), columns=[f"dest_x_enc_{i}" for i in range(2 * num_freqs)])
dest_y_enc_df = pd.DataFrame(np.stack(dest_y_enc.values), columns=[f"dest_y_enc_{i}" for i in range(2 * num_freqs)])

# concatenate these encoding dataframes with the original dataframe
trips_test = pd.concat(
    [trips_test.reset_index(drop=True), start_x_enc_df, start_y_enc_df, dest_x_enc_df, dest_y_enc_df], axis=1
)

In [30]:
trips_train

,vehicle_id,start_x,start_y,destination_x,destination_y,manhattan_distance,euclidean_distance,hour_of_day,vehicle_type_bus,vehicle_type_car,...,dest_x_enc_6,dest_x_enc_7,dest_y_enc_0,dest_y_enc_1,dest_y_enc_2,dest_y_enc_3,dest_y_enc_4,dest_y_enc_5,dest_y_enc_6,dest_y_enc_7
0,veh0,266.91,1136.32,2495.35,381.74,2983.02,2352.729438,0.0,1.0,0.0,...,-0.847181,0.435431,-0.999334,-0.072920,0.145453,0.287811,0.036484,-0.997338,0.989365,0.957687
1,veh1,433.71,1323.05,2238.10,224.42,2903.02,2112.536662,0.0,0.0,1.0,...,0.418446,-0.649806,-0.979289,0.396544,-0.728068,-0.998188,-0.202465,-0.918016,0.685505,-0.060165
2,veh10,554.95,600.75,2140.55,947.22,1932.07,1623.012268,0.0,1.0,0.0,...,-0.209220,-0.912454,-0.999556,-0.059595,0.118977,0.236264,0.029811,-0.998223,0.992897,0.971689
3,veh100,1796.03,1429.95,2007.19,234.24,1406.87,1214.212069,0.0,0.0,1.0,...,0.407736,-0.667502,0.981749,-0.373422,0.692818,0.999199,-0.190182,-0.927662,0.721113,0.040007
4,veh1000,2386.95,638.63,2457.40,452.88,256.20,198.661181,9.0,0.0,0.0,...,-0.903293,0.631875,0.471206,0.831230,0.924210,-0.705884,0.882023,0.555929,-0.381885,-0.708327
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15373,veh9995,2096.93,186.71,1948.17,259.32,221.37,165.534739,6.0,0.0,0.0,...,0.040566,-0.996709,0.990410,-0.273675,0.526454,0.895186,-0.138163,-0.961822,0.850204,0.445693
15374,veh9996,2095.41,771.68,555.30,991.85,1760.28,1555.767862,6.0,0.0,1.0,...,-0.995586,0.982382,-0.779131,-0.976814,0.418253,-0.759824,0.626861,-0.214091,-0.908330,0.650128
15375,veh9997,2043.88,1166.53,2630.04,980.08,772.61,615.099299,6.0,0.0,1.0,...,-0.511560,-0.476613,-0.096756,-0.192605,-0.377997,-0.699904,0.995308,0.981276,0.925807,0.714237
15376,veh9998,2502.36,423.10,2365.00,1427.68,1141.94,1013.927387,6.0,0.0,1.0,...,-0.787213,0.239408,0.984924,0.340757,-0.640727,-0.983860,0.172986,-0.940151,0.767769,0.178939
